## TUGAS PRAKTIKUM 6

nama = Cahyo Adi Nugroho

NPM = 2505060034

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Tugas6").master("local[*]").getOrCreate()

26/09/27 22:25:01 WARN Utils: Your hostname, cahyoalim-ThinkPad-L14-Gen-2 resolves to a loopback address: 127.0.1.1; using 192.168.18.76 instead (on interface wlp9s0)
26/09/27 22:25:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/27 22:25:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
print("--==SOAL 1==--")

transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)

produk = spark.read.json("tugas6_produk.json")

ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)


print("--==data transaksi")
print(transaksi.count())
transaksi.printSchema()
transaksi.show()
print("")

print("--==data produk")
print(produk.count())
produk.printSchema()
produk.show()
print("")

print("--==data ulasan")
print(ulasan.count())
ulasan.printSchema()
ulasan.show()



--==SOAL 1==--
--==data transaksi
5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

+--------+----------+------------+-------------------+
|order_id|product_id|unit_terjual|            tanggal|
+--------+----------+------------+-------------------+
|     TX0|        21|           2|2026-10-10 00:00:00|
|     TX1|         8|           6|2026-10-25 00:00:00|
|     TX2|        25|           4|2026-10-13 00:00:00|
|     TX3|         3|           4|2026-10-18 00:00:00|
|     TX4|        19|           6|2026-10-07 00:00:00|
|     TX5|        10|           3|2026-10-01 00:00:00|
|     TX6|        26|           6|2026-10-24 00:00:00|
|     TX7|         4|           1|2026-10-03 00:00:00|
|     TX8|         7|           2|2026-10-09 00:00:00|
|     TX9|        30|           1|2026-10-27 00:00:00|
|    TX10|        20|           1|2026-10-04 00:00:00|
|    TX11|     

In [5]:
print("--==SOAL 2==--")

from pyspark.sql.functions import col, when


gabungan = transaksi.join(ulasan, on="order_id", how="left")

gabungan = gabungan.join(produk, on="product_id", how="inner")

gabungan = gabungan.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

gabungan.show()

--==SOAL 2==--
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|    

In [6]:
print("--==SOAL 3==--")

gabungan = gabungan.withColumn("ada_ulasan", when(col("rating").isNotNull(), True).otherwise(False))

hasil = gabungan.fillna({"rating" : 0})

hasil.show()

--==SOAL 3==--
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      true|
|         8|     TX1|           6|2026-10-25 00:00:00|     0|250000|     Fashion|   Produk-8|         1500000|     false|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      true|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      true|
|        19|     TX4|           6|2026-10-07 00:00:00|     0| 75000|Rumah Tangga|  Produk-19|          450000|     false|
|        

In [21]:
print("--==SOAL 4==--")

!hdfs dfs -mkdir -p /user/mahasiswa/tugas6

hdfs_path = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

hasil.write.mode("overwrite").partitionBy("kategori").parquet(hdfs_path)

!hdfs dfs -ls /user/mahasiswa/tugas6/hasil_etl

baca_hdfs = spark.read.parquet(hdfs_path)
print(baca_hdfs.count())
baca_hdfs.show()

--==SOAL 4==--


Found 6 items
-rw-r--r--   3 cahyoalim supergroup          0 2026-09-24 11:23 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - cahyoalim supergroup          0 2026-09-24 11:23 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
drwxr-xr-x   - cahyoalim supergroup          0 2026-09-24 11:23 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
drwxr-xr-x   - cahyoalim supergroup          0 2026-09-24 11:23 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
drwxr-xr-x   - cahyoalim supergroup          0 2026-09-24 11:23 /user/mahasiswa/tugas6/hasil_etl/kategori=Makanan
drwxr-xr-x   - cahyoalim supergroup          0 2026-09-24 11:23 /user/mahasiswa/tugas6/hasil_etl/kategori=Rumah Tangga
5000
+----------+--------+------------+-------------------+------+------+-----------+----------------+----------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|nama_produk|total_pendapatan|ada_ulasan|  kategori|
+----------+--------+------------+-------------------+-

In [25]:
print("--==SOAL E==--")
from pyspark.sql.functions import col, count, when, round

E = hasil.groupBy("kategori").agg(count("order_id").alias("total_transaksi"), count(when(col("ada_ulasan") == True, 1)).alias("berulasan"))

jawaban_E = E.withColumn("persentase_ulasan", round((col("berulasan") / col("total_transaksi"))*100, 2)).orderBy("persentase_ulasan")

jawaban_E.show()

--==SOAL E==--
+------------+---------------+---------+-----------------+
|    kategori|total_transaksi|berulasan|persentase_ulasan|
+------------+---------------+---------+-----------------+
|     Makanan|            536|      369|            68.84|
|   Kesehatan|            961|      662|            68.89|
|     Fashion|           1035|      726|            70.14|
|Rumah Tangga|            815|      575|            70.55|
|  Elektronik|           1653|     1168|            70.66|
+------------+---------------+---------+-----------------+



Kategori produk Makanan memiliki persentase transaksi berulasan paling rendah, yaitu sebesar 68,84% atau tepatnya 369 ulasan dari 536 transaksi. Hal ini penting diketahui oleh tim marketing karena ulasan dan rating dari pembeli  sangat krusial untuk membangun tingkat kepercayaan calon pelanggan baru pada produk makanan. Informasi ini menjadi acuan bagi tim marketing untuk membuat strategi  partisipasi, seperti memberikan insentif poin atau koin promosi bagi pelanggan yang memberikan ulasan di kategori tersebut.